In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.tools import aggregate_results

In [3]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def highlight_extrema(s, use_italics=False, exclude_index=None):

    comp_list = s
    if exclude_index is not None:
        comp_list = s.drop(labels=exclude_index)
    
    styles = []
    for v in s:      
        style = ''
        if v not in comp_list.values:
            pass
        elif v == comp_list.max():
            style = 'font-weight: bold;'
        elif v == comp_list.min():
            style = 'font-weight: bold; font-style: italic;'
        styles.append(style)
    
    return styles

def percentage_improvement(df, against="0-non-expl"):
    if against in ["", "max"] or against is None:
        return df.apply(lambda col: (col - col.max()) / col.max(), axis=0)

    if against == "min":
        return df.apply(lambda col: (col.max() - col.min()) / col.min(), axis=0) 
        
    return df.apply(lambda col: (col.loc[col.index != against].max() - col.loc[against]) / col.loc[against], axis=0)
    

def diff_to_baseline(df, t="max", perc=False):

    against = "0-non-expl"
    
    if t == "max":
        fun = pd.Series.max
    else:
        fun = pd.Series.min

    return df.apply(lambda col: (fun(col.loc[col.index != against]) - col.loc[against]) / (1 if not perc else col.loc[against]), axis=0)



def value_improvement(df, against="0-non-expl"):
    if against in ["", "max"] or against is None:
        return df.apply(lambda col: (col - col.max()), axis=0)

    if against == "min":
        return df.apply(lambda col: col.max() - col.min(), axis=0) 
        
    
    return df.apply(lambda col: col.loc[col.index != against].max() - col.loc[against], axis=0)   

In [14]:
### Load F1 data

metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]

res = pd.read_csv("./data/metrics.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

In [15]:
### Create top 10 suites for each label

metric_cols = ["Theme", "Topic", "Concept", "Sum"]

f1s = res[["model", "suite"] + metric_cols_f1]
f1s = f1s.rename(columns={col: col.split(" ")[0] for col in metric_cols_f1})

f1s["sum"] = f1s["theme"] + f1s["topic"] + f1s["concept"]
f1s = f1s.rename(columns={col: capitalize(col) for col in f1s.columns})


top10s = [
    f1s
        .sort_values(by=col, ascending=False)[:10]
    for col in metric_cols
]

for df in top10s:
    pass
    #print(df.to_latex(index=False, float_format="%.3f"))

In [18]:
### Create one table per label, where best of each model is presented

tops_per_model = [
    f1s
        .loc[f1s.groupby('Model').idxmax()[col]]
        .sort_values(by="Model", ascending=True)
    for col in metric_cols
]

for df in tops_per_model:
    pass
    #print(df.to_latex(index=False, float_format="%.3f"))

In [7]:
### Baseline, top score for each model and label

baselines = f1s["Suite"] == "0-non-expl"

max_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmax()

best_per_label = pd.concat(
    [
        f1s[baselines],
        f1s.loc[max_values.values.reshape(-1)]
    ]
).groupby(["Model", "Suite"]).max()

model_level = best_per_label.index.names.index('Model')
models = best_per_label.index.get_level_values(model_level)
models = pd.Series(list(models)).unique()

for model in models:
    pass

    per_model = best_per_label.loc[model]
    per_model.loc["Difference"] = value_improvement(per_model)
    per_model.loc["Difference (\\%)"] = percentage_improvement(per_model) 
    
    #print("%" + model)
    #print(per_model.style.apply(lambda col: highlight_extrema(col)).to_latex(convert_css=True))


In [22]:
### Table with models ordered by their score per label

models_ordered_by_scores = {}

for i, col in enumerate(metric_cols):
    best_scores_for_col = tops_per_model[i]
    ordered = best_scores_for_col.sort_values(by=col, ascending=False)
 
    models_ordered_by_scores[col] = ordered["Model"].to_list()
    models_ordered_by_scores[f"{col} Difference"] = value_improvement(ordered.loc[:, [col]], None)[col].to_list()

#print(pd.DataFrame(models_ordered_by_scores).to_latex(index=False))

In [10]:
### Baseline, top score for each model and label

baselines = f1s["Suite"] == "0-non-expl"

f1s_grouped = f1s.loc[:, f1s.columns != "Suite"].groupby(by=["Model"])

# Indices of best/worst scores per label
max_values = f1s_grouped.idxmax()
min_values = f1s_grouped.idxmin()

extreme_values = max_values.join(min_values, on="Model", lsuffix=" Max", rsuffix=" Min")

df = extreme_values.copy()
df.columns = df.columns.str.split(' ', expand=True)

# Rearrange: stack Min/Max into rows
out = (
    df.stack(level=1)  # moves Min/Max to rows
      .rename_axis(index=['Model', 'Type'])  # rename index levels
      .reset_index(level=1)  # bring Type into column
)

out.index = [out.index, out['Type']]
out = out.drop(columns='Type')
out.index.names = ['Model', '']

suites = out.map(lambda e: f1s.loc[e, "Suite"])
values = out.apply(lambda col: f1s.loc[col.values, col.name].values)

joined = values.join(suites, on=["Model", ""], rsuffix=" Suite")

for model in models:
    per_model = joined.loc[model].copy()

    differences = per_model.apply(lambda col: col.max() - col.min() if "Suite" not in col.name else "")

    per_model.loc["Difference"] = differences
    per_model = per_model.reindex(sorted(per_model.columns), axis=1)

    #print(f"% {model}")
    #print(per_model.to_latex())

/tmp/ipykernel_2294394/1628760773.py:16: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(level=1)  # moves Min/Max to rows


In [24]:
### Best and worst score for each model and label

max_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmin()
min_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmax()

best_per_label = pd.concat(
    [
        f1s.loc[max_values.values.reshape(-1)],
        f1s.loc[min_values.values.reshape(-1)],
    ]
).groupby(["Model", "Suite"]).max()

for model in models:
    per_model = best_per_label.loc[model]

    per_model.style.apply(
        lambda col: highlight_extrema(col, True) 
    )

    differences = value_improvement(per_model, "min")
    as_percentage = percentage_improvement(per_model, "min")
    
    per_model.loc["Difference"] = differences
    per_model.loc["Difference (\\%)"] = as_percentage
        
    print("%" + model)
    print(
        per_model.style.apply(lambda col: highlight_extrema(col, True, exclude_index=["Difference", "Difference (\\%)"])).to_latex(convert_css=True)
    )

%Llama-70B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
0-non-expl & 0.944742 & \bfseries 0.977860 & 0.857651 & 2.780253 \\
1-neg-expl & \bfseries \itshape 0.892189 & \bfseries \itshape 0.922404 & 0.831153 & 2.645746 \\
1-pos-impl & 0.931374 & 0.955997 & \bfseries \itshape 0.434863 & \bfseries \itshape 2.322233 \\
6-pos-expl & \bfseries 0.948763 & 0.972256 & \bfseries 0.874916 & \bfseries 2.795934 \\
Difference & 0.056573 & 0.055456 & 0.440053 & 0.473701 \\
Difference (\%) & 0.063410 & 0.060121 & 1.011936 & 0.203985 \\
\end{tabular}

%Llama-8B
\begin{tabular}{lrrrr}
 & Theme & Topic & Concept & Sum \\
Suite &  &  &  &  \\
1-neg-expl & 0.570127 & \bfseries \itshape 0.189303 & 0.741252 & \bfseries \itshape 1.500682 \\
1-neg-impl & \bfseries \itshape 0.467009 & 0.542374 & 0.699036 & 1.708419 \\
1-pos-impl & 0.717053 & 0.743152 & \bfseries \itshape 0.635414 & 2.095619 \\
6-mix-expl & \bfseries 0.895698 & 0.820526 & \bfseries 0.902410 & 2.618634 \\
6-pos-e

In [25]:
### Baseline, max, min for each model and label

baselines = f1s["Suite"] == "0-non-expl"

max_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmax()
min_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmin()

best_per_label = pd.concat(
    [
        f1s[baselines],
        f1s.loc[max_values.values.reshape(-1)],
        f1s.loc[min_values.values.reshape(-1)],
    ]
).groupby(["Model", "Suite"]).max()

for model in models:
    pass

    per_model = best_per_label.loc[model]
    
    #print("%" + model)
    #print(
    #    per_model.style.apply(
    #        lambda col: highlight_extrema(
    #            col,
    #            True,
    #            #exclude_index=[idx for idx in per_model.index if "Diff" in idx]
    #        )
    #    ).to_latex(convert_css=True)
    #)

In [26]:
### Differences: max to baseline, min to baseline, max to min

for model in models:
    pass

    per_model = best_per_label.loc[model]

    best = diff_to_baseline(per_model, "max", False)
    best_p = diff_to_baseline(per_model, "max", True)

    worst = diff_to_baseline(per_model, "min", False)
    worst_p = diff_to_baseline(per_model, "min", True)

    rel = value_improvement(per_model, "min")
    rel_p = percentage_improvement(per_model, "min")

    diffs = pd.DataFrame(
        {
            "Diff hi/bl": best,
            "Diff hi/bl (\\%)": best_p,
            "Diff lo/bl": worst,
            "Diff lo/bl (\\%)": worst_p,
            "Diff hi/lo": rel,
            "Diff hi/lo (\\%)": rel_p,
        }
    )
    
    #print("%" + model)
    #print(
    #    diffs.T.to_latex()
    #)